# Low-level `padeldescriptor` (no RDKit / pandas)

This notebook uses a **minimal** padelpy2 install: Python stdlib + system Java + the stock Yap JAR. It does **not** import RDKit or pandas and does **not** use `Calculator`.

Install: `pip install padelpy2` (not `padelpy2[calc]`).

Import the wrapper submodule directly. Avoid `import padelpy2` followed by `Calculator` if you want to keep the Calculator stack unloaded; catalogs and `padeldescriptor` are available without optional extras.

## Setup

Requirements: Python 3.9+, **Java JRE 8+** on `PATH`, and `padelpy2` without optional extras.

In [1]:
from csv import DictReader
from pathlib import Path
from tempfile import TemporaryDirectory

from padelpy2.wrapper import PADEL_PATH, padeldescriptor

assert Path(PADEL_PATH).is_file(), "vendored PaDEL-Descriptor.jar missing"

## Motivation

[padelpy](https://github.com/ecrl/padelpy) popularized a thin CLI over the stock JAR. padelpy2 keeps that surface as `padeldescriptor` for file/directory workflows. Use this path when you already have structure files (`.smi`, SDF, MOL) and do not want RDKit or DataFrames.

Engine: Yap, C. W. (2011). PaDEL-Descriptor. *J. Comput. Chem.* 32(7), 1466–1474.

## Minimal example

Write a small SMILES file and compute **1D/2D** descriptors into a CSV. Read a few columns with the stdlib `csv` module.

In [2]:
with TemporaryDirectory() as tmp:
    root = Path(tmp)
    smi_path = root / "mols.smi"
    out_path = root / "descriptors.csv"
    smi_path.write_text("CCO ethanol\nc1ccccc1 benzene\n", encoding="utf-8")

    result = padeldescriptor(
        mol_dir=str(smi_path),
        d_file=str(out_path),
        d_2d=True,
        threads=1,
        retainorder=True,
        headless=True,
    )
    assert result is not None
    with open(result, newline="", encoding="utf-8") as handle:
        rows = list(DictReader(handle))

print("n_mols", len(rows))
print("n_columns", len(rows[0]))
# Stock JAR includes MW under the Weight descriptor family (column name MW).
for row in rows:
    print(row.get("Name"), "MW=", row.get("MW"))

n_mols 2
n_columns 1445
ethanol MW= 46.041864812
benzene MW= 78.046950192


## Fingerprints

Set `fingerprints=True` (and typically leave `d_2d=False` if you only want fingerprints) to append fingerprint columns. Full 2D + fingerprints produces a wide CSV.

In [3]:
with TemporaryDirectory() as tmp:
    root = Path(tmp)
    smi_path = root / "one.smi"
    out_path = root / "fp.csv"
    smi_path.write_text("CCO ethanol\n", encoding="utf-8")
    result = padeldescriptor(
        mol_dir=str(smi_path),
        d_file=str(out_path),
        d_2d=False,
        fingerprints=True,
        threads=1,
        headless=True,
    )
    with open(result, newline="", encoding="utf-8") as handle:
        row = next(DictReader(handle))
    fp_cols = [c for c in row if c != "Name"]
print("fingerprint columns", len(fp_cols))
print("sample keys", fp_cols[:5])

fingerprint columns 881
sample keys ['PubchemFP0', 'PubchemFP1', 'PubchemFP2', 'PubchemFP3', 'PubchemFP4']


## Aromaticity flag

Default `detectaromaticity=False` matches classic stock-JAR behavior. Pass `detectaromaticity=True` when you want PaDEL to re-detect aromaticity before descriptor calculation (same keyword as padelpy).

In [4]:
with TemporaryDirectory() as tmp:
    root = Path(tmp)
    smi_path = root / "benzene.smi"
    out_default = root / "default.csv"
    out_detect = root / "detect.csv"
    smi_path.write_text("c1ccccc1 benzene\n", encoding="utf-8")
    common = dict(mol_dir=str(smi_path), d_2d=True, threads=1, headless=True)
    padeldescriptor(d_file=str(out_default), detectaromaticity=False, **common)
    padeldescriptor(d_file=str(out_detect), detectaromaticity=True, **common)
    with open(out_default, newline="", encoding="utf-8") as handle:
        r0 = next(DictReader(handle))
    with open(out_detect, newline="", encoding="utf-8") as handle:
        r1 = next(DictReader(handle))
print("default naAromAtom", r0.get("naAromAtom"), "nAromBond", r0.get("nAromBond"))
print("detect  naAromAtom", r1.get("naAromAtom"), "nAromBond", r1.get("nAromBond"))

default naAromAtom 6 nAromBond 6
detect  naAromAtom 6 nAromBond 6


## Interpretation

* Outputs are engine CSV files; you choose how to load them (`csv`, pandas if installed separately, etc.).
* For RDKit `Mol` objects and DataFrames, install `padelpy2[calc]` and see `examples/example.ipynb`.
* Prefer padelpy when you need stdlib-only *and* SMILES→dict helpers without this package’s catalogs.

## Takeaways

- Minimal install: `pip install padelpy2` + Java.
- Import `from padelpy2.wrapper import padeldescriptor`.
- Pass structure files/directories and read the CSV yourself.
- Use `padelpy2[calc]` + the Calculator notebook when you want RDKit → DataFrame.

## Further reading

- Sphinx: installation, when-to-use, migration
- Yap (2011) PaDEL-Descriptor DOI: https://doi.org/10.1002/jcc.21707
- RDKit Calculator tutorial: `examples/example.ipynb`